## Import

In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:4'

%load_ext autoreload
%autoreload 2

## Dataset

In [2]:
from datasets.Student_t import MultivariateStudentT

d = 64
dim_x = dim_y = d//2
rho = 0.7
df = 2
mean = np.zeros(dim_x + dim_y)
V = np.eye(dim_x)*rho
dispersion = np.eye(dim_x + dim_y)
dispersion[0:dim_x, :][:, dim_x:] = V
dispersion[dim_x:, :][:, 0:dim_x] = V


dataset = MultivariateStudentT(
        dim_x=dim_x,
        dim_y=dim_y,
        mean=mean,
        dispersion=dispersion,
        df=df)

X, Y = dataset.sample(10000)
X, Y = torch.Tensor(X).float().to(device), torch.Tensor(Y).float().to(device)

print(f'MI is {dataset.mutual_information(): 0.2f}')
print('X', X.shape, 'Y', Y.shape)

MI is  12.00
X torch.Size([10000, 32]) Y torch.Size([10000, 32])


## Hyperaparams

In [5]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.K_components = 6
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## MI estimate

In [20]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(None, None, None, hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.9847890138626099 loss val= 1.248490333557129 best val loss= 1.248490333557129 best t= 0
finished: t= 76 loss= 0.6544008851051331 loss val= 0.43409568071365356 best val loss= 0.4217236638069153 best t= 68
finished: t= 152 loss= 0.6363645195960999 loss val= 0.5359927415847778 best val loss= 0.4017622172832489 best t= 133
finished: t= 228 loss= 0.5631267428398132 loss val= 0.40305274724960327 best val loss= 0.4017622172832489 best t= 133
finished: t= 304 loss= 0.40758517384529114 loss val= 0.5158026218414307 best val loss= 0.4017622172832489 best t= 133
finished: t= 380 loss= 0.5872576236724854 loss val= 0.41924428939819336 best val loss= 0.4017622172832489 best t= 133
finished: t= 456 loss= 0.6024556159973145 loss val= 0.41377198696136475 best val loss= 0.4017622172832489 best t= 133
finished: t= 532 loss= 0.42532801628112793 loss val= 0.4955471158027649 best val loss= 0.4017622172832489 best t= 133
finished: t= 608 loss= 0.39626455307006836 loss val= 0.41963756084

In [21]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.014106690883636475 loss val= 0.007076414301991463 best val loss= 0.007076414301991463 best t= 0
finished: t= 76 loss= -4.802267551422119 loss val= -5.379001617431641 best val loss= -5.552684783935547 best t= 68
finished: t= 152 loss= -6.560861587524414 loss val= -4.024857044219971 best val loss= -6.032134056091309 best t= 151
finished: t= 228 loss= -5.933433532714844 loss val= -5.933506965637207 best val loss= -6.180631637573242 best t= 209
finished: t= 304 loss= -6.180723190307617 loss val= -5.633354187011719 best val loss= -6.248440742492676 best t= 302
finished: t= 380 loss= -5.688346862792969 loss val= -6.116384983062744 best val loss= -6.474769115447998 best t= 376
finished: t= 456 loss= -6.370297431945801 loss val= -6.171544075012207 best val loss= -6.474769115447998 best t= 376
finished: t= 532 loss= -5.503018379211426 loss val= -4.2643303871154785 best val loss= -6.573030471801758 best t= 475
finished: t= 608 loss= -5.88701057434082 loss val= -6.094004154

In [6]:
## Vector copula estimation
from estimators.VCE import VCE

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))


K components= 6 copula transform= True
nde type: FM
finished: t= 0 loss= 1.5365803241729736 loss val= 1.8037937879562378 best val loss= 1.8037937879562378 best t= 0
finished: t= 126 loss= 1.396256685256958 loss val= 1.811636209487915 best val loss= 1.3126180171966553 best t= 10


finished: t= 0 loss= 1.7377551794052124 loss val= 1.9109890460968018 best val loss= 1.9109890460968018 best t= 0
finished: t= 126 loss= 0.8632340431213379 loss val= 1.3544737100601196 best val loss= 1.1452319622039795 best t= 111
finished: t= 252 loss= 0.8558151721954346 loss val= 1.26667058467865 best val loss= 1.0996003150939941 best t= 180
finished: t= 378 loss= 0.7712759375572205 loss val= 1.2510846853256226 best val loss= 1.0876296758651733 best t= 330
finished: t= 504 loss= 0.8816654086112976 loss val= 1.230973482131958 best val loss= 1.0802024602890015 best t= 380


finished: t= 0 loss= 455.7710876464844 loss val= 454.69805908203125 best val loss= 454.69805908203125 best t= 0
finished: t= 101 loss= 78.5